# 05 - Credit Risk Modelling

## Objective
Train and compare Logistic Regression, Random Forest and XGBoost; handle class
imbalance with evidence rather than habit; select a model and a decision threshold
on the validation set; and report a single, final, unbiased evaluation on the test set.

## Split discipline
| Split | Used for |
|---|---|
| **Train** | fitting preprocessing + estimators |
| **Validation** | model selection, imbalance strategy, threshold tuning |
| **Test** | one final evaluation, never for any decision |

The dataset has no origination date, so a chronological out-of-time split is not
possible; we use a stratified random split with a fixed seed and note the limitation.

In [1]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import numpy as np
import pandas as pd

from src import config
from src.data import preprocessing
from src.evaluation import metrics as M
from src.features.engineering import add_credit_risk_features, get_feature_lists
from src.models import train as T
from src.risk.segmentation import RiskBands, segment_portfolio, summarise_segments

%matplotlib inline

## Configuration

In [3]:
RANDOM_STATE = config.RANDOM_STATE
np.random.seed(RANDOM_STATE)
print("test size:", config.TEST_SIZE, "| validation size:", config.VALIDATION_SIZE)

test size: 0.2 | validation size: 0.2


## Data loading

In [4]:
raw = preprocessing.load_raw_data()
TARGET = preprocessing.detect_target(raw)
clean, _ = preprocessing.clean_data(raw, TARGET)
df = add_credit_risk_features(clean)
df.shape

(32416, 25)

## Train / validation / test split

In [5]:
splits = T.split_data(df, TARGET)
X_train, X_val, X_test = splits["X_train"], splits["X_val"], splits["X_test"]
y_train, y_val, y_test = splits["y_train"], splits["y_val"], splits["y_test"]

pd.DataFrame({
    "rows": [len(X_train), len(X_val), len(X_test)],
    "default_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"]).round(4)

,rows,default_rate
train,20745,0.2187
validation,5187,0.2186
test,6484,0.2187


## Preprocessing pipeline

`ColumnTransformer` inside a `Pipeline`:

* numeric -> median imputation (+ standard scaling for the linear model only)
* categorical -> most-frequent imputation + one-hot encoding with `handle_unknown="ignore"`

Because these steps are fitted inside the pipeline, they see the training fold only.

In [6]:
numeric, categorical = get_feature_lists(df, TARGET, drop=list(T.REDUNDANT_COLUMNS))
preprocessor = T.build_preprocessor(numeric, categorical)
preprocessor.fit(X_train)
print(len(T.build_preprocessor(numeric, categorical).fit(X_train)
          .get_feature_names_out()), "columns after encoding")

49 columns after encoding


## Class imbalance

About 22% of the loans default. That is imbalanced but not extreme, so the minority
class is well represented in absolute terms (thousands of cases).

**SMOTE is not used.** It would fabricate borrowers, and the synthetic minority
points distort the predicted probabilities -- which is exactly what the risk bands
depend on. Instead both variants are fitted and compared on the validation set:
estimator re-weighting (`class_weight` / `scale_pos_weight`) versus training on the
natural distribution.

In [7]:
scale_pos_weight = T.compute_scale_pos_weight(y_train)
print(f"negatives/positives = {scale_pos_weight:.2f}")

study = {}
fitted_by_strategy = {}
for strategy, weighting in (("class-weighted", True), ("unweighted", False)):
    models = T.build_models(numeric, categorical, scale_pos_weight, class_weighting=weighting)
    fitted_by_strategy[strategy] = T.fit_models(models, X_train, y_train)
    for name, pipeline in fitted_by_strategy[strategy].items():
        study[(name, strategy)] = M.evaluate_predictions(y_val, T.predict_proba(pipeline, X_val))

pd.DataFrame(study).T.round(4)

negatives/positives = 3.57


,,threshold,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Brier
Logistic Regression,class-weighted,0.5,0.8230,0.5663,0.8131,0.6676,0.8909,0.7541,0.1306
Random Forest,class-weighted,0.5,0.9090,0.8152,0.7549,0.7839,0.9251,0.8718,0.0813
XGBoost,class-weighted,0.5,0.9107,0.8014,0.7866,0.7939,0.9453,0.8995,0.0705
Logistic Regression,unweighted,0.5,0.8776,0.7807,0.6120,0.6861,0.8899,0.7646,0.0936
Random Forest,unweighted,0.5,0.9242,0.9469,0.6922,0.7998,0.9232,0.8721,0.0655
XGBoost,unweighted,0.5,0.9360,0.9663,0.7328,0.8335,0.9452,0.9008,0.0539


Pick, per model, the variant with the better validation PR-AUC (ties broken on the
Brier score, i.e. probability quality).

In [8]:
fitted, chosen = {}, {}
for name in fitted_by_strategy["unweighted"]:
    ranked = sorted(
        ["class-weighted", "unweighted"],
        key=lambda s: (study[(name, s)]["PR-AUC"], -study[(name, s)]["Brier"]),
        reverse=True,
    )
    chosen[name] = ranked[0]
    fitted[name] = fitted_by_strategy[ranked[0]][name]
chosen

{'Logistic Regression': 'unweighted',
 'Random Forest': 'unweighted',
 'XGBoost': 'unweighted'}

## Model comparison on the validation set

In [9]:
val_proba = {name: T.predict_proba(pipeline, X_val) for name, pipeline in fitted.items()}
val_metrics = {name: M.evaluate_predictions(y_val, proba) for name, proba in val_proba.items()}
val_table = M.comparison_table(val_metrics)
val_table

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
XGBoost,0.9360,0.9663,0.7328,0.8335,0.9452,0.9008
Random Forest,0.9242,0.9469,0.6922,0.7998,0.9232,0.8721
Logistic Regression,0.8776,0.7807,0.6120,0.6861,0.8899,0.7646


In [10]:
M.plot_roc_pr_curves(y_val, val_proba, filename="nb_roc_pr_validation.png")
M.plot_calibration(y_val, val_proba, filename="nb_calibration.png")

WindowsPath('D:/Downloads/ML-Driven-Credit-Risk-Loan-Default-Prediction-System-main/credit-risk-loan-default-ml/reports/figures/nb_calibration.png')

## Threshold tuning (validation only)

Accuracy is not the objective: predicting "nobody defaults" would already score
about 78%. Two policies are derived:

1. **Balanced** -- maximise F1 for the default class.
2. **High recall** -- maximise precision subject to recall >= 85%, the shape a
   lender uses when a missed default costs far more than a rejected good customer.

In [11]:
best_name = val_table.index[0]
threshold, threshold_metrics = M.tune_threshold(y_val, val_proba[best_name], objective="f1")
policy_threshold, policy_metrics = M.tune_threshold(
    y_val, val_proba[best_name], objective="max_precision_at_recall", min_recall=0.85)

print(f"selected model: {best_name}")
print(f"balanced threshold   = {threshold:.2f} -> {pd.Series(threshold_metrics).round(4).to_dict()}")
print(f"high-recall threshold = {policy_threshold:.2f} -> {pd.Series(policy_metrics).round(4).to_dict()}")

selected model: XGBoost
balanced threshold   = 0.46 -> {'threshold': 0.46, 'Accuracy': 0.9364, 'Precision': 0.96, 'Recall': 0.7399, 'F1': 0.8357, 'ROC-AUC': 0.9452, 'PR-AUC': 0.9008, 'Brier': 0.0539}
high-recall threshold = 0.17 -> {'threshold': 0.17, 'Accuracy': 0.8818, 'Precision': 0.682, 'Recall': 0.8607, 'F1': 0.761, 'ROC-AUC': 0.9452, 'PR-AUC': 0.9008, 'Brier': 0.0539}


## Final evaluation on the test set

This is the only place the test set is touched.

In [12]:
test_proba = {name: T.predict_proba(pipeline, X_test) for name, pipeline in fitted.items()}
test_metrics = {
    name: M.evaluate_predictions(y_test, proba,
                                 threshold if name == best_name else 0.5)
    for name, proba in test_proba.items()
}
test_table = M.comparison_table(test_metrics)
test_table

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
XGBoost,0.9369,0.9658,0.7377,0.8365,0.9430,0.9005
Random Forest,0.9266,0.9564,0.6961,0.8057,0.9190,0.8707
Logistic Regression,0.8740,0.7770,0.5945,0.6736,0.8861,0.7588


In [13]:
print(M.classification_text_report(y_test, test_proba[best_name], threshold))
M.plot_confusion_matrix(y_test, test_proba[best_name], threshold,
                        title=f"{best_name} - test", filename="nb_confusion_matrix.png")
M.plot_probability_distribution(y_test, test_proba[best_name], filename="nb_pd_distribution.png")

                 precision    recall  f1-score   support

Non-default (0)     0.9311    0.9927    0.9609      5066
    Default (1)     0.9658    0.7377    0.8365      1418

       accuracy                         0.9369      6484
      macro avg     0.9485    0.8652    0.8987      6484
   weighted avg     0.9387    0.9369    0.9337      6484



WindowsPath('D:/Downloads/ML-Driven-Credit-Risk-Loan-Default-Prediction-System-main/credit-risk-loan-default-ml/reports/figures/nb_pd_distribution.png')

## Probability of default and risk segmentation

`predict_proba` gives the PD directly; the bands are project-defined policy
thresholds (`LOW_RISK_THRESHOLD`, `HIGH_RISK_THRESHOLD` in `src/config.py`), not
statistical constants. The check that matters is whether the observed default rate
inside each band tracks the predicted PD.

In [14]:
bands = RiskBands()
print(bands.describe())
scored = segment_portfolio(X_test.assign(**{TARGET: y_test}), test_proba[best_name],
                           bands, target=TARGET)
summarise_segments(scored, bands, target=TARGET)

{'Low Risk': 'PD < 10%', 'Medium Risk': '10% <= PD < 40%', 'High Risk': 'PD >= 40%'}


,customers,share_of_portfolio,avg_predicted_pd,total_exposure,avg_loan_amount,actual_default_rate,defaulted_exposure
risk_category,,,,,,,
Low Risk,3997,0.6164,0.0359,37165475,9298.3425,0.0330,1246100
Medium Risk,1361,0.2099,0.1789,12328775,9058.6150,0.1609,1945725
High Risk,1126,0.1737,0.9071,12755725,11328.3526,0.9476,12085350


In [15]:
scored[["person_income", "loan_amnt", "loan_grade", "default_probability",
        "risk_category", TARGET]].head(10)

,person_income,loan_amnt,loan_grade,default_probability,risk_category,loan_status
21156,56461.0,5400,E,0.277859,Medium Risk,0
26854,108000.0,4925,B,0.049783,Low Risk,1
26333,102000.0,18000,B,0.001309,Low Risk,0
24056,45000.0,10000,B,0.010493,Low Risk,0
24947,30000.0,12000,B,0.990539,High Risk,1
28808,113000.0,9600,C,0.103352,Medium Risk,0
28361,73000.0,16000,F,0.940333,High Risk,1
8425,90000.0,7500,D,0.107735,Medium Risk,0
29094,135000.0,24250,C,0.127036,Medium Risk,0
30192,52000.0,8000,A,0.042883,Low Risk,0


## Persisting the artefacts

In [16]:
T.save_model(fitted[best_name])
print("Saved to", config.MODEL_FILE)
print("Run `python run_pipeline.py` to regenerate every artefact, including "
      "models/feature_config.json, reports/metrics.json and the figures.")

Saved to D:\Downloads\ML-Driven-Credit-Risk-Loan-Default-Prediction-System-main\credit-risk-loan-default-ml\models\credit_risk_model.pkl
Run `python run_pipeline.py` to regenerate every artefact, including models/feature_config.json, reports/metrics.json and the figures.


## Results
See the two comparison tables above -- validation for the decision, test for the
final numbers. The same figures are written to `reports/metrics.json` by the pipeline.

## Interpretation
Gradient boosting wins on ranking quality (ROC-AUC / PR-AUC) and on probability
quality (Brier). Logistic regression stays in the project as a transparent
benchmark: the gap between it and the boosted model is the price of interpretability.

## Conclusion
The selected pipeline is saved with its preprocessing attached, so serving applies
exactly the training-time transformations. Explainability follows in notebook 06.